In [1]:
# Встановлення Ultralytics та його залежностей у середовищі ноутбука
%pip install -q ultralytics

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Стандартна бібліотека Python
from collections import Counter
from pathlib import Path
import os
import random
import shutil
import warnings

# Робота з даними та конфігурацією
import numpy as np
import pandas as pd
import yaml

# Обробка та візуалізація зображень
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from PIL import Image, ImageDraw

# Глибоке навчання та детекція об'єктів
import torch
from ultralytics import YOLO

# Налаштування відображення
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU не знайдено — обчислення виконуватимуться на CPU.")

PyTorch version: 2.13.0+cpu
CUDA available: False
GPU не знайдено — обчислення виконуватимуться на CPU.


## 5. Конфігурація YOLO

Створимо конфігураційний файл датасету та завантажимо попередньо навчену модель YOLOv9s.

In [3]:
# У Colab датасет розташований у /content/data.
# Локально використовується поточна директорія проєкту.
if Path("/content/data").exists():
    dataset_root = Path("/content/data")
else:
    dataset_root = Path.cwd().resolve()

class_names = [
    "door",
    "cabinetDoor",
    "refrigeratorDoor",
    "window",
    "chair",
    "table",
    "cabinet",
    "couch",
    "openedDoor",
    "pole",
]

dataset_config = {
    "path": str(dataset_root),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(class_names),
    "names": class_names,
}

# Створюємо або оновлюємо наявний data.yaml
yaml_path = dataset_root / "data.yaml"

with yaml_path.open("w", encoding="utf-8") as file:
    yaml.safe_dump(
        dataset_config,
        file,
        sort_keys=False,
        allow_unicode=True,
    )

print(f"Файл конфігурації: {yaml_path}")
print(f"Коренева директорія датасету: {dataset_root}")
print(f"Кількість класів: {len(class_names)}")

print("\nВміст data.yaml:")
with yaml_path.open("r", encoding="utf-8") as file:
    print(file.read())

Файл конфігурації: D:\IT\WOOLF\WoolfUni\Deep Learning for CV and NLP\HW8\data.yaml
Коренева директорія датасету: D:\IT\WOOLF\WoolfUni\Deep Learning for CV and NLP\HW8
Кількість класів: 10

Вміст data.yaml:
path: D:\IT\WOOLF\WoolfUni\Deep Learning for CV and NLP\HW8
train: train/images
val: valid/images
test: test/images
nc: 10
names:
- door
- cabinetDoor
- refrigeratorDoor
- window
- chair
- table
- cabinet
- couch
- openedDoor
- pole



In [4]:
required_directories = {
    "train": dataset_root / "train" / "images",
    "valid": dataset_root / "valid" / "images",
    "test": dataset_root / "test" / "images",
}

for split_name, split_path in required_directories.items():
    if split_path.exists():
        image_count = sum(
            1
            for file in split_path.iterdir()
            if file.suffix.lower() in {".jpg", ".jpeg", ".png"}
        )

        print(
            f"{split_name}: директорію знайдено, "
            f"кількість зображень — {image_count}"
        )
    else:
        print(f"{split_name}: директорію не знайдено — {split_path}")

train: директорію знайдено, кількість зображень — 1011
valid: директорію знайдено, кількість зображень — 229
test: директорію знайдено, кількість зображень — 107


In [5]:
MODEL_NAME = "yolov9s.pt"

model = YOLO(MODEL_NAME)

print(f"Обрана модель: {MODEL_NAME}")
print(f"Тип задачі: {model.task}")

Обрана модель: yolov9s.pt
Тип задачі: detect


## 6. Навчання моделі YOLOv9s

Навчимо попередньо навчену модель YOLOv9s на датасеті Indoor Objects Detection та збережемо найкращі ваги.

In [6]:
# Вибір пристрою
device = 0 if torch.cuda.is_available() else "cpu"

# Автоматичний batch для GPU, невеликий batch для CPU
batch_size = -1 if torch.cuda.is_available() else 4

# У Windows workers=0 допомагає уникнути проблем із multiprocessing
workers_count = 2 if os.name != "nt" else 0

# Директорія для результатів
project_dir = Path.cwd() / "runs" / "indoor_detection"

print(f"Пристрій: {device}")
print(f"Batch size: {batch_size}")
print(f"Workers: {workers_count}")
print(f"Результати будуть збережені в: {project_dir}")

if not torch.cuda.is_available():
    print(
        "\nУВАГА: GPU не знайдено. "
        "Навчання YOLOv9s на CPU може тривати дуже довго."
    )

Пристрій: cpu
Batch size: 4
Workers: 0
Результати будуть збережені в: d:\IT\WOOLF\WoolfUni\Deep Learning for CV and NLP\HW8\runs\indoor_detection

УВАГА: GPU не знайдено. Навчання YOLOv9s на CPU може тривати дуже довго.


In [7]:
train_results = model.train(
    # Дані
    data=str(yaml_path),

    # Основні параметри
    epochs=100,
    imgsz=640,
    batch=batch_size,
    device=device,
    workers=workers_count,

    # Early stopping
    patience=20,

    # Навчання з pretrained-ваг
    pretrained=True,
    optimizer="auto",

    # Відтворюваність результатів
    seed=42,
    deterministic=True,

    # Прискорення навчання на GPU
    amp=torch.cuda.is_available(),

    # Валідація та графіки
    val=True,
    plots=True,

    # Збереження результатів
    save=True,
    save_period=10,
    project=str(project_dir),
    name="yolov9s_baseline",
    exist_ok=False,

    # Виведення прогресу
    verbose=True,
)

Ultralytics 8.4.131  Python-3.10.6 torch-2.13.0+cpu CPU (Intel Core i5-8350U 1.70GHz)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\IT\WOOLF\WoolfUni\Deep Learning for CV and NLP\HW8\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov9s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.

KeyboardInterrupt: 